In [1]:
# Monte Carlo Dropout version
# visibility classification problem using other parameters
%tensorflow_version 2.x

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

import tensorflow as tf

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.layers import Input, Lambda, Dense, Reshape, Flatten, \
    LSTM, Concatenate, Activation, BatchNormalization
from tensorflow.keras.models import Model, load_model

from tensorflow.keras import losses
from tensorflow.keras import backend as K

from sklearn import metrics

from sklearn.model_selection import KFold

tf.test.gpu_device_name()
print(tf.__version__)

2.3.0


In [4]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [5]:
# Load the data
obs_data_vis = np.load('/content/gdrive/My Drive/Colab Notebooks/Visobs.npy', allow_pickle=True)
model_data_vis = np.load('/content/gdrive/My Drive/Colab Notebooks/Vismodel.npy', allow_pickle=True)

obs_data_T = np.load('/content/gdrive/My Drive/Colab Notebooks/Tobs.npy', allow_pickle=True)
model_data_T = np.load('/content/gdrive/My Drive/Colab Notebooks/Tmodel.npy', allow_pickle=True)

obs_data_Td = np.load('/content/gdrive/My Drive/Colab Notebooks/Tdobs.npy', allow_pickle=True)
model_data_Td = np.load('/content/gdrive/My Drive/Colab Notebooks/Tdmodel.npy', allow_pickle=True)

In [6]:
# Max model visibility data is 9999 so truncate model vis to that
model_data_vis[model_data_vis > np.max(obs_data_vis)] = np.max(obs_data_vis) 

In [7]:
# Bin the data
bins = [0, 150, 350, 600, 800, 1500, 3000, 5000, 10000]
obs_data_vis = pd.cut(obs_data_vis, bins, labels=[0,1,2,3,4,5,6,7])
model_binned = np.zeros(model_data_vis.shape)
for i in range(4):
  model_binned[i,:] = pd.cut(model_data_vis[i,:], bins, labels=[0,1,2,3,4,5,6,7])
model_data_vis = model_binned

In [8]:
# Normalize the Td/T data
# obs_data_T = (obs_data_T-np.mean(obs_data_T))/np.std(obs_data_T)
# obs_data_Td = (obs_data_Td-np.mean(obs_data_Td))/np.std(obs_data_Td)

# model_data_T = (model_data_T-np.mean(model_data_T))/np.std(model_data_T)
# model_data_Td = (model_data_Td-np.mean(model_data_Td))/np.std(model_data_Td)

In [9]:
# Use the past_history nr of data to predict future_target nr of data
past_history = 6
future_target = 3

In [13]:
np.concatenate((model_data_vis, np.reshape(obs_data_vis,(1,-1))), axis=0).duplicated()

ValueError: ignored

In [10]:
# Add obs data to model data
dataset = np.concatenate((model_data_vis, np.reshape(obs_data_vis,(1,-1))), axis=0)
dataset_vis = np.swapaxes(dataset,0,1)

dataset = np.concatenate((model_data_T, np.reshape(obs_data_T,(1,-1))), axis=0)
dataset_T = np.swapaxes(dataset,0,1)

dataset = np.concatenate((model_data_Td, np.reshape(obs_data_Td,(1,-1))), axis=0)
dataset_Td = np.swapaxes(dataset,0,1)

dataset_vis.shape

ValueError: ignored

In [ ]:
# Plot part of data to make sure concatenation and normalization was successful
plt.plot(dataset_vis[:72,0])
plt.plot(dataset_vis[:72,1])
plt.plot(dataset_vis[:72,3])
plt.plot(dataset_vis[:72,4])
plt.legend(['p1','p2','p4','obs_old'])


In [ ]:
def multivariate_data(dataset, target, start_index, end_index, history_size,
                      target_size, step, single_step=False):
  data = []
  labels = []

  start_index = start_index + history_size
  if end_index is None:
    end_index = len(dataset) - target_size

  for i in range(start_index, end_index):
    #indices = range(i-history_size, i, step)
    indices = range(i-history_size, i+target_size, step)
    data.append(dataset[indices])

    if single_step:
      labels.append(target[i+target_size])
    else:
      labels.append(target[i:i+target_size])

  return np.array(data), np.array(labels)

In [ ]:
STEP = 1

# This is the visibility (target) dataset
x_vis, y_vis = multivariate_data(dataset_vis, dataset_vis[:,4], 0,
                                                 None, past_history,
                                                 future_target, STEP)

# These are the auxiliary (T, Td) datasets
x_T, _ = multivariate_data(dataset_T, dataset_T[:,4], 0,
                                                 None, past_history,
                                                 future_target, STEP)

x_Td, _ = multivariate_data(dataset_Td, dataset_Td[:,4], 0,
                                                 None, past_history,
                                                 future_target, STEP)

In [ ]:
# Set the future obs to the average of the model data
x_vis[:,past_history:,4] = np.round(np.average(x_vis[:,past_history:,0:4], axis=2))

x_T[:,past_history:,4] = np.round(np.average(x_T[:,past_history:,0:4], axis=2))
x_Td[:,past_history:,4] = np.round(np.average(x_Td[:,past_history:,0:4], axis=2))

In [ ]:
# Using T-Td as feature
x_TTd = x_T-x_Td
x_TTd = (x_TTd-np.mean(x_TTd))/np.std(x_TTd)
x_T = (x_T-np.mean(x_T))/np.std(x_T)

In [ ]:
print(x_vis.shape)
print(y_vis.shape)

In [ ]:
# Data shape
print ('Single window of past history : {}'.format(x_vis[0].shape))
print ('\n Target visibility to predict : {}'.format(y_vis[0].shape))

In [ ]:
# K-Fold Cross Validator
num_folds = 10
kfold = KFold(n_splits=num_folds, shuffle=True)

In [ ]:
# Define per-fold score containers
persistence_per_fold = []
averaged_per_fold = []
ml_per_fold = []

In [ ]:
# Compute PC, bias and POD from confusion matrices
def compute_PC_Bias_POD(y_true, y_obs):

  labels = np.uint32(['0','1','2','3','4','5','6','7'])
  confusion_matrix = metrics.confusion_matrix(y_true, y_obs, labels=labels)
  # print(confusion_matrix)

  # Proportion correct
  PC = np.trace(confusion_matrix)/len(y_true)

  Bias = np.zeros(8)
  POD = np.zeros(8)

  # Bias and POD
  for j in range(8):
    Bias[j] = np.sum(confusion_matrix[j,:])/np.sum(confusion_matrix[:,j])
    POD[j] = confusion_matrix[j,j]/np.sum(confusion_matrix[:,j])

  return PC, Bias, POD

In [ ]:
input_shape = (past_history+future_target, 5,)
EPOCHS = 10
BATCH_SIZE = 32

In [ ]:
PC_pers = np.zeros(future_target) 
Bias_pers = np.zeros((future_target, 8)) 
POD_pers = np.zeros((future_target, 8)) 

PC_aver = np.zeros(future_target) 
Bias_aver = np.zeros((future_target, 8)) 
POD_aver = np.zeros((future_target, 8)) 

PC_ml = np.zeros(future_target) 
Bias_ml = np.zeros((future_target, 8)) 
POD_ml = np.zeros((future_target, 8))

In [ ]:
fold_no = 1

for train, test in kfold.split(x_vis, y_vis):

  # Visibility branch
  in_vis = Input(shape=input_shape)
  b_vis = LSTM(128, return_sequences=True, dropout=0.)(in_vis)

  # T/Td branch
  in_TTd = Input(shape=input_shape)
  b_TTd = LSTM(128, return_sequences=True, dropout=0.)(in_TTd)

  # T branch
  in_T = Input(shape=input_shape)
  b_T = LSTM(128, return_sequences=True, dropout=0.)(in_T)

  # Now concatenate with visibility branch
  b = Concatenate()([b_vis, b_TTd])

  b = LSTM(128, return_sequences=True, dropout=0.)(b)
  b = LSTM(128, dropout=0.)(b)
  out = Dense(future_target, activation='relu')(b)

  model = Model([in_vis, in_TTd, in_T], out)
  model.compile(optimizer='adam', loss='mae',
                metrics=['mae'])

  # Get current train and test data set
  x_train_vis = x_vis[train]
  x_train_TTd = x_TTd[train]
  x_train_T = x_T[train]
  y_train_vis = y_vis[train]

  x_test_vis = x_vis[test]
  x_test_TTd = x_TTd[test]
  x_test_T = x_T[test]
  y_test_vis = y_vis[test]

  # Keep only 10% of cases where there has been a constant (max) visibility
  idx = np.sum(y_train_vis, axis=1) != np.max(np.sum(y_train_vis, axis=1))
  idx_2 = np.sum(y_train_vis, axis=1) == np.max(np.sum(y_train_vis, axis=1))

  x_train_vis_1 = x_train_vis[idx,];  x_train_vis_2 = x_train_vis[idx_2,]
  n_c = np.int(0.1*len(x_train_vis_2))
  y_train_vis_1 = y_train_vis[idx,]; y_train_vis_2 = y_train_vis[idx_2,]
  x_train_TTd_1 = x_train_TTd[idx,]; x_train_TTd_2 = x_train_TTd[idx_2,]
  x_train_T_1 = x_train_T[idx,]; x_train_T_2 = x_train_T[idx_2,]

  x_train_vis = np.concatenate((x_train_vis_1, x_train_vis_2[:n_c]))
  x_train_TTd = np.concatenate((x_train_TTd_1, x_train_TTd_2[:n_c]))
  x_train_T = np.concatenate((x_train_T_1, x_train_T_2[:n_c]))
  y_train_vis = np.concatenate((y_train_vis_1, y_train_vis_2[:n_c]))
  
  idx_v = np.sum(y_test_vis, axis=1) != np.max(np.sum(y_test_vis, axis=1))
  x_test_vis = x_test_vis[idx_v,]
  y_test_vis = y_test_vis[idx_v,]
  x_test_TTd = x_test_TTd[idx_v,]
  x_test_T = x_test_T[idx_v,]

  # Train model
  multi_step_history = model.fit([x_train_vis, x_train_TTd, x_train_T], 
                                 y_train_vis, batch_size=BATCH_SIZE,
                                 epochs=EPOCHS)

  # Now verify the model
  n_val = len(x_test_vis)

  pers = np.zeros((n_val, future_target))
  aver = np.zeros((n_val, future_target))
  ml = np.zeros((n_val, future_target))

  err_pers = np.zeros(n_val)
  err_aver = np.zeros(n_val)
  err_ml = np.zeros(n_val)

  for i in range(n_val):

    # 1) Persistence forecast
    pers[i,] = x_test_vis[i, past_history-1, 4]*np.ones(future_target)
    err_pers[i] = np.mean(np.square(pers - y_test_vis[i,]))

    # 2) Averaged model forecast
    aver[i,] = x_test_vis[i, past_history:, 4]
    err_aver[i] = np.mean(np.square(aver - y_test_vis[i,]))

    # 3) ML forecast
    vis = np.expand_dims(x_test_vis[i,], axis=0)
    TTd = np.expand_dims(x_test_TTd[i,], axis=0)
    T = np.expand_dims(x_test_T[i,], axis=0)
    ml[i,] = np.round(model([vis, TTd, T])[0,])
    err_ml[i] = np.mean(np.square(ml[0,] - y_test_vis[i,]))

  # Verification based on forecast horizon    
  for i in range(future_target):
  
    # Forecasts at hour i
    y_pers = pers[:, i]
    y_aver = aver[:, i]
    y_ml = ml[:, i]

    # True value at hour i
    y_true = y_test_vis[:, i]

    # Persistence
    PC_curr, Bias_curr, POD_curr = compute_PC_Bias_POD(y_true, y_pers)
    PC_pers[i] += PC_curr
    Bias_pers[i,] += Bias_curr
    POD_pers[i,] += POD_curr

    # Averaged model
    PC_curr, Bias_curr, POD_curr = compute_PC_Bias_POD(y_true, y_aver)
    PC_aver[i] += PC_curr
    Bias_aver[i,] += Bias_curr
    POD_aver[i,] += POD_curr

    # ML model
    PC_curr, Bias_curr, POD_curr = compute_PC_Bias_POD(y_true, y_ml)
    PC_ml[i] += PC_curr
    Bias_ml[i,] += Bias_curr
    POD_ml[i,] += POD_curr

  # Save error scores
  persistence_per_fold.append(np.mean(err_pers))
  averaged_per_fold.append(np.mean(err_aver))
  ml_per_fold.append(np.mean(err_ml))

  fold_no += 1

In [ ]:
# Average the results
PC_pers = PC_pers/num_folds
PC_aver = PC_aver/num_folds
PC_ml = PC_ml/num_folds

Bias_pers = Bias_pers/num_folds
Bias_aver = Bias_aver/num_folds
Bias_ml = Bias_ml/num_folds

for i in range(future_target):
  print('PC correct at forecast hour {} (persistence, averaged model, ML model): {}, {}, {}'.format(i+1, PC_pers[i], PC_aver[i], PC_ml[i]))
  #print('Bias correct at forecast hour {} (persistence, averaged model, ML model): {}, {}, {}'.format(i+1, Bias_pers[i], Bias_aver[i], Bias_ml[i]))

In [ ]:
# 0.52, 0.44, 0.40

In [ ]:
# Average scores
print('Errors per fold')
for i in range(0, len(ml_per_fold)):
  print(f'> Fold {i+1} - Persistence: {persistence_per_fold[i]} - Averaged model: {averaged_per_fold[i]} - ML model: {ml_per_fold[i]}')
print('Average scores for all folds:')
print(f'> Persistence: {np.mean(persistence_per_fold)} (+- {np.std(persistence_per_fold)})')
print(f'> Averaged model: {np.mean(averaged_per_fold)} (+- {np.std(averaged_per_fold)})')
print(f'> ML model: {np.mean(ml_per_fold)} (+- {np.std(ml_per_fold)})')

In [ ]:
# Average scores for all folds:
# > Persistence: 432.1666666666667 (+- 54.22284266493844)
# > Averaged model: 1001.6666666666666 (+- 95.32307637130101)
# > ML model: 352.11761506470793 (+- 62.49230402406637)